**ReadME**  
**V1: 25 Historial Quarters Data (5 Assets)**  

**Overview of the Entire JupyterNotebook**  
Section 1: Required Libraries  
Section 2: Data Processing
* 2.1 Single Asset Data Processing  
* 2.2 Single Asset Test Case  
* 2.3 Multi Asset Data Processing  
* 2.4 Multi Asset Test Case

Section 3: LSTM Machine Learning Estimator  
* 3.1 Pre-Processing
* 3.2 Pre-Processing Test Case
* 3.3 LSTM Model SetUp
* 3.4 LSTM Test Case

**Use Intruction**  
Please run all the sections with functions to construct the framework, Test Case Sections are optional to run

**Section 1: Required Libraries**

In [1]:
import pandas as pd
import yfinance as yf
import numpy as np

from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Dense
from tensorflow.keras.regularizers import l2

**Section 2: Data Processing**  
Section 2.1 Single Asset Data Processing

In [31]:
def get_historical_returns(ticker, start_date, end_date, frequency="monthly"):
    'Function to fetch Historical Price data and compute returns'

    data = yf.download(ticker,start=start_date, end=end_date)

    # Calculate Daily Returns
    daily_data = data.copy()
    daily_data['Return'] = daily_data['Close'].pct_change()
    daily_returns = daily_data[['Return']].dropna()

    # Calculate Monthly Returns
    monthly_data = data.copy()
    monthly_data['Return'] = monthly_data['Close']
    monthly_data = monthly_data['Return'].resample('M').last()
    monthly_returns = monthly_data.pct_change() 
    monthly_returns = monthly_returns.dropna()
    
    if frequency == "daily": return daily_returns
    if frequency == "monthly": return monthly_returns

    return monthly_data

def resample_quaterly_data(quaterly_data, target_data):
    'Repeat the quaterly available ratios to same frequency as target return'
    
    quaterly_data.index = pd.to_datetime(quaterly_data.index)
    target_data.index = pd.to_datetime(target_data.index)

    # Resample the quaterly data to daily frequency using Forward Fill
    quaterly_data.index = quaterly_data.index + pd.DateOffset(days=1)
    aligned_quaterly_data = quaterly_data.reindex(target_data.index, method='ffill')

    aligned_quaterly_data = aligned_quaterly_data.dropna()
    return aligned_quaterly_data


def load_features(ticker, start_date, end_date):
    'Function to Load all features for a single company'

    # Load the Excel file and read Data from the file
    file_path = ticker + '-US.xlsx'
    sheet_name = ticker + '-US'           
    data = pd.read_excel(file_path, sheet_name=sheet_name, engine='openpyxl')

    # Remove rows with any NaN values
    data = data.dropna()
    # Reset the index of the DataFrame and drop the old index
    data = data.reset_index(drop=True)

    data = data.set_index('Date').T
    data.index = pd.to_datetime(data.index, format='%b \'%y')
    data.index = data.index + pd.offsets.MonthEnd()
    ratio_data = data.apply(pd.to_numeric)

    # Select a few columns
    pe_column = 'Price/Earnings'  
    pb_column = 'Price/Book Value'  
    roa_column = 'Return on Assets' 
    roe_column = 'Return on Equity ' 
    fcf_column = 'Free Cash Flow per Share'
    ratio_data = data[[pe_column, pb_column, roa_column, roe_column, fcf_column]]
    
    # Process Return Data
    returns_data = get_historical_returns(ticker, start_date, end_date)
    adjusted_ratio_data = resample_quaterly_data(ratio_data, returns_data)

    features = pd.concat([adjusted_ratio_data, returns_data],axis=1)

    return features

Section 2.2 Single Asset Test Case

In [12]:
ticker = 'MSFT'
start_date = '2017-08-30'
end_date = '2023-09-30'
msft_test = load_features(ticker, start_date, end_date)
msft_return_test = pd.DataFrame(msft_test['Return'])

[*********************100%%**********************]  1 of 1 completed


In [13]:
print(msft_test)

                   0         1         2         3         4    Return
Date                                                                  
2017-09-30  0.050556  0.000000  0.298605  0.390735  0.007562 -0.003745
2017-10-31  1.000000  0.213520  0.000000  0.000000  0.045929  0.116660
2017-11-30  1.000000  0.213520  0.000000  0.000000  0.045929  0.011902
2017-12-31  1.000000  0.213520  0.000000  0.000000  0.045929  0.016277
2018-01-31  0.774905  0.261287  0.070215  0.099281  0.042825  0.110708
...              ...       ...       ...       ...       ...       ...
2023-05-31  0.331155  0.626558  0.795709  0.687703  0.843734  0.068769
2023-06-30  0.331155  0.626558  0.795709  0.687703  0.843734  0.036999
2023-07-31  0.200204  0.450937  0.825133  0.696146  0.954834 -0.013567
2023-08-31  0.200204  0.450937  0.825133  0.696146  0.954834 -0.024292
2023-09-30  0.200204  0.450937  0.825133  0.696146  0.954834 -0.036643

[73 rows x 6 columns]


In [14]:
print(msft_return_test)

              Return
Date                
2017-09-30 -0.003745
2017-10-31  0.116660
2017-11-30  0.011902
2017-12-31  0.016277
2018-01-31  0.110708
...              ...
2023-05-31  0.068769
2023-06-30  0.036999
2023-07-31 -0.013567
2023-08-31 -0.024292
2023-09-30 -0.036643

[73 rows x 1 columns]


Section 2.3 Multi Asset Data Processing

In [32]:
def multi_df(ticker_list, start_date, end_date):
    company_data = {}
    for ticker in ticker_list:
        company_data[ticker] = load_features(ticker, start_date, end_date)

    # Initialize a list to hold DataFrames with the new multi-index
    multi_index_dfs = []

    for company, df in company_data.items():
        # Set the company name as an additional level in the index
        df_multi_index = df.copy()
        df_multi_index['Company'] = company
        df_multi_index.set_index(['Company', df_multi_index.index], inplace=True)
        
        # Append to the list
        multi_index_dfs.append(df_multi_index)

    # Concatenate all DataFrames into a single multi-index DataFrame
    final_df = pd.concat(multi_index_dfs)
    

    return final_df

Section 2.4 Multi Asset Test Case
Final Output Below:  
1.multi_df: multi-index feature set (xvalue)  
2.multi_return: multi-index return set (yvalue)

In [33]:
# Loading Phase: Took a while to run this (Don't Rerun)
stock_tickers_test1 = ["AAPL", "MSFT", "GOOG", "GOOGL", "AMZN"]
start_date = '2017-08-30'
end_date = '2023-09-30'
final_df = multi_df(stock_tickers_test1, start_date, end_date)

[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed


In [34]:
multi_df_test1 = final_df
multi_return_test1 = pd.DataFrame(multi_df_test1['Return'])

In [35]:
print(multi_df_test1)

                    Price/Earnings  Price/Book Value  Return on Assets  \
Company Date                                                             
AAPL    2017-09-30       16.752174          5.893825         13.873932   
        2017-10-31       17.392600          6.133904         13.693618   
        2017-11-30       17.392600          6.133904         13.693618   
        2017-12-31       17.392600          6.133904         13.693618   
        2018-01-31       16.194981          6.536859         15.189578   
...                            ...               ...               ...   
AMZN    2023-05-31      103.666004          7.973825          2.913516   
        2023-06-30      103.666004          7.973825          2.913516   
        2023-07-31       66.374269          7.176739          4.387678   
        2023-08-31       66.374269          7.176739          4.387678   
        2023-09-30       66.374269          7.176739          4.387678   

                    Return on Equity 

In [36]:
print(multi_return_test1)

                      Return
Company Date                
AAPL    2017-09-30 -0.060244
        2017-10-31  0.096808
        2017-11-30  0.016623
        2017-12-31 -0.015246
        2018-01-31 -0.010636
...                      ...
AMZN    2023-05-31  0.143480
        2023-06-30  0.081108
        2023-07-31  0.025468
        2023-08-31  0.032391
        2023-09-30 -0.078907

[365 rows x 1 columns]


**Section 3: LSTM Machine Learning Estimator**  
Section 3.1 Pre-Processing

In [37]:
def create_sequences(features, targets, seq_length):
    'Function to create sequence'
    'Need to define the sequence length: e.g. using 4 quaters to predict the next quater'

    xs = []
    ys = []

    for i in range(len(features)-seq_length):
        x = features[i:(i+seq_length)]
        y = targets.iloc[i+seq_length]
        xs.append(x)
        ys.append(y)
        
    return np.array(xs), np.array(ys)

Section 3.2 Pre-Processing Test Case

In [23]:
# Single Asset Test Case
features = msft_test
targets = msft_return_test
seq_length = 6
msft_X_test, msft_y_test = create_sequences(features, targets, seq_length)

In [24]:
print(msft_X_test)

[[[ 0.05055576  0.          0.29860472  0.39073457  0.00756242
   -0.0037448 ]
  [ 1.          0.2135196   0.          0.          0.04592866
    0.11665999]
  [ 1.          0.2135196   0.          0.          0.04592866
    0.01190187]
  [ 1.          0.2135196   0.          0.          0.04592866
    0.01627662]
  [ 0.77490495  0.26128719  0.07021549  0.09928057  0.04282453
    0.11070845]
  [ 0.77490495  0.26128719  0.07021549  0.09928057  0.04282453
   -0.01305132]]

 [[ 1.          0.2135196   0.          0.          0.04592866
    0.11665999]
  [ 1.          0.2135196   0.          0.          0.04592866
    0.01190187]
  [ 1.          0.2135196   0.          0.          0.04592866
    0.01627662]
  [ 0.77490495  0.26128719  0.07021549  0.09928057  0.04282453
    0.11070845]
  [ 0.77490495  0.26128719  0.07021549  0.09928057  0.04282453
   -0.01305132]
  [ 0.77490495  0.26128719  0.07021549  0.09928057  0.04282453
   -0.02666098]]

 [[ 1.          0.2135196   0.          0.      

In [25]:
print(msft_y_test)

[[-0.02666098]
 [ 0.02465213]
 [ 0.05688623]
 [-0.00232695]
 [ 0.07575298]
 [ 0.0589178 ]
 [ 0.01816078]
 [-0.06610129]
 [ 0.03819869]
 [-0.08404725]
 [ 0.02815793]
 [ 0.07277601]
 [ 0.05275376]
 [ 0.10734275]
 [-0.05298626]
 [ 0.08311777]
 [ 0.01724393]
 [ 0.01166798]
 [ 0.00848686]
 [ 0.03121626]
 [ 0.0558695 ]
 [ 0.04174919]
 [ 0.07945465]
 [-0.04828762]
 [-0.0265415 ]
 [ 0.13632616]
 [ 0.02254335]
 [ 0.11055932]
 [ 0.00737065]
 [ 0.1000927 ]
 [-0.06739679]
 [-0.03736985]
 [ 0.05729247]
 [ 0.03900589]
 [ 0.04289187]
 [ 0.00181065]
 [ 0.01458817]
 [ 0.06960168]
 [-0.00991355]
 [ 0.08498879]
 [ 0.05171654]
 [ 0.05956267]
 [-0.06611896]
 [ 0.17629107]
 [-0.00310596]
 [ 0.01733268]
 [-0.0753449 ]
 [-0.03919867]
 [ 0.03186181]
 [-0.09986705]
 [-0.02035887]
 [-0.05532059]
 [ 0.09309662]
 [-0.06863999]
 [-0.10926686]
 [-0.00330609]
 [ 0.09912546]
 [-0.06004543]
 [ 0.03331661]
 [ 0.00649692]
 [ 0.1558816 ]
 [ 0.06576491]
 [ 0.06876913]
 [ 0.03699867]
 [-0.01356667]
 [-0.02429151]
 [-0.03664

In [38]:
# Multi Asset Test Case
features = multi_df_test1
targets = multi_return_test1
seq_length = 6
multi_X_test1, multi_y_test1 = create_sequences(features, targets, seq_length)

In [39]:
print(multi_X_test1)

[[[ 1.67521740e+01  5.89382500e+00  1.38739320e+01  3.68675080e+01
    2.46677700e+00 -6.02439322e-02]
  [ 1.73926000e+01  6.13390400e+00  1.36936180e+01  3.70704610e+01
    2.56447200e+00  9.68076735e-02]
  [ 1.73926000e+01  6.13390400e+00  1.36936180e+01  3.70704610e+01
    2.56447200e+00  1.66233609e-02]
  [ 1.73926000e+01  6.13390400e+00  1.36936180e+01  3.70704610e+01
    2.56447200e+00 -1.52459138e-02]
  [ 1.61949810e+01  6.53685900e+00  1.51895780e+01  4.08629680e+01
    2.67806400e+00 -1.06364303e-02]
  [ 1.61949810e+01  6.53685900e+00  1.51895780e+01  4.08629680e+01
    2.67806400e+00  6.38475955e-02]]

 [[ 1.73926000e+01  6.13390400e+00  1.36936180e+01  3.70704610e+01
    2.56447200e+00  9.68076735e-02]
  [ 1.73926000e+01  6.13390400e+00  1.36936180e+01  3.70704610e+01
    2.56447200e+00  1.66233609e-02]
  [ 1.73926000e+01  6.13390400e+00  1.36936180e+01  3.70704610e+01
    2.56447200e+00 -1.52459138e-02]
  [ 1.61949810e+01  6.53685900e+00  1.51895780e+01  4.08629680e+01
    

In [41]:
print(multi_y_test1)

[[-0.05805073]
 [-0.01501969]
 [ 0.13076365]
 [-0.00941828]
 [ 0.02798332]
 [ 0.19622688]
 [-0.00830294]
 [-0.03047756]
 [-0.18404459]
 [-0.11669838]
 [ 0.05515403]
 [ 0.04031478]
 [ 0.09702572]
 [ 0.05643591]
 [-0.12757259]
 [ 0.13051916]
 [ 0.07639448]
 [-0.02018395]
 [ 0.07296156]
 [ 0.11068444]
 [ 0.07432869]
 [ 0.09878389]
 [ 0.05400993]
 [-0.11679759]
 [-0.06976146]
 [ 0.15537377]
 [ 0.08216479]
 [ 0.14738625]
 [ 0.16513164]
 [ 0.21437974]
 [-0.10252632]
 [-0.06001206]
 [ 0.09360649]
 [ 0.1145737 ]
 [-0.00550151]
 [-0.08108521]
 [ 0.0073396 ]
 [ 0.07621781]
 [-0.05210715]
 [ 0.09910927]
 [ 0.06498243]
 [ 0.04092967]
 [-0.06803663]
 [ 0.05865727]
 [ 0.10347129]
 [ 0.0742287 ]
 [-0.01571216]
 [-0.0552695 ]
 [ 0.05747339]
 [-0.09713079]
 [-0.05588327]
 [-0.08142969]
 [ 0.18863365]
 [-0.0325518 ]
 [-0.120977  ]
 [ 0.10955137]
 [-0.03462891]
 [-0.12227255]
 [ 0.11052106]
 [ 0.02162319]
 [ 0.1186486 ]
 [ 0.02898726]
 [ 0.04461344]
 [ 0.09433005]
 [ 0.01278546]
 [-0.04367525]
 [-0.08867

Section 3.3 LSTM Model SetUp

In [42]:
# LSTM Model Set Up

# Model architecture
model = tf.keras.Sequential([
    LSTM(512, return_sequences=True),
    Dropout(0.02),
    LSTM(256, return_sequences=True),
    LSTM(128),
    Dense(1, activation='linear', kernel_regularizer=l2(0.0005))
])

# Compile the model
optimizer = tf.keras.optimizers.Adam(learning_rate=0.005)
model.compile(optimizer=optimizer, loss='mean_squared_error')

Section 3.4 LSTM Test Case

In [43]:
# Variable in Use and Constant Define
X = multi_X_test1
y = multi_y_test1
train_ratio = 0.8
num_epoch = 500
num_batch = 12

# Split data into training and validation sets

train_size = int(len(X) * train_ratio)
X_train, X_vali = X[:train_size], X[train_size:]
y_train, y_vali = y[:train_size], y[train_size:]

# Train the model
model.fit(X_train, y_train, epochs=num_epoch, batch_size=num_batch, validation_data=(X_vali, y_vali))


Epoch 1/500
24/24 [==============================] - 5s 47ms/step - loss: 0.7890 - val_loss: 0.0137
Epoch 2/500
24/24 [==============================] - 0s 17ms/step - loss: 0.0067 - val_loss: 0.0104
Epoch 3/500
24/24 [==============================] - 0s 16ms/step - loss: 0.0069 - val_loss: 0.0104
Epoch 4/500
24/24 [==============================] - 0s 16ms/step - loss: 0.0071 - val_loss: 0.0104
Epoch 5/500
24/24 [==============================] - 0s 16ms/step - loss: 0.0068 - val_loss: 0.0107
Epoch 6/500
24/24 [==============================] - 0s 18ms/step - loss: 0.0074 - val_loss: 0.0110
Epoch 7/500
24/24 [==============================] - 0s 17ms/step - loss: 0.0069 - val_loss: 0.0105
Epoch 8/500
24/24 [==============================] - 0s 16ms/step - loss: 0.0069 - val_loss: 0.0111
Epoch 9/500
24/24 [==============================] - 0s 17ms/step - loss: 0.0067 - val_loss: 0.0116
Epoch 10/500
24/24 [==============================] - 0s 17ms/step - loss: 0.0069 - val_loss: 0.0104

In [44]:
# Evaluate the model
X_test, y_test = X_vali, y_vali

loss = model.evaluate(X_test, y_test)
print(f"Mean Squared Error: {loss}")

3/3 [==============================] - 1s 7ms/step - loss: 0.0110
Mean Squared Error: 0.011018265038728714


In [45]:
# Make predictions
X_pred, y_pred = X_vali, y_vali

predictions = model.predict(X_pred)
print(predictions)

3/3 [==============================] - 1s 6ms/step
[[ 0.03703988]
 [ 0.06481124]
 [ 0.01720517]
 [-0.01528406]
 [ 0.05167422]
 [ 0.03806003]
 [ 0.03881908]
 [ 0.00593953]
 [ 0.01844042]
 [ 0.04256942]
 [ 0.03854533]
 [ 0.06938019]
 [ 0.03838482]
 [ 0.03727682]
 [ 0.0486335 ]
 [ 0.03773908]
 [ 0.07188313]
 [ 0.03904589]
 [ 0.06117402]
 [ 0.03050187]
 [ 0.02973701]
 [ 0.0485156 ]
 [ 0.01910196]
 [ 0.02470965]
 [ 0.05721813]
 [ 0.03683593]
 [ 0.02384969]
 [ 0.03112783]
 [ 0.02515819]
 [ 0.02530673]
 [ 0.02461444]
 [ 0.02820253]
 [ 0.04051363]
 [ 0.04075572]
 [ 0.03144021]
 [ 0.05113103]
 [ 0.0496321 ]
 [ 0.06406359]
 [ 0.04738731]
 [ 0.04422855]
 [ 0.06320947]
 [ 0.03735607]
 [ 0.06020122]
 [ 0.03004878]
 [ 0.03470971]
 [ 0.03682715]
 [ 0.01866013]
 [ 0.00306493]
 [ 0.02997046]
 [ 0.06226739]
 [ 0.03582705]
 [ 0.06960157]
 [ 0.02664506]
 [ 0.02954695]
 [ 0.03216521]
 [ 0.03281994]
 [ 0.02693391]
 [ 0.02420098]
 [ 0.03110651]
 [ 0.03831407]
 [ 0.03663787]
 [ 0.05682474]
 [ 0.02972854]
 [ 0

In [148]:
print(y_pred)

[[-0.01963079]
 [ 0.14971652]
 [ 0.06466238]
 [-0.00618657]
 [ 0.24063898]
 [ 0.04242906]
 [-0.04304937]
 [ 0.0820748 ]
 [ 0.04053941]
 [ 0.04306519]
 [ 0.04567601]
 [ 0.13236448]
 [-0.00482431]
 [-0.20219175]
 [ 0.05767175]
 [-0.1113497 ]
 [ 0.14431709]
 [-0.04590598]
 [ 0.08593571]
 [ 0.08185875]
 [-0.0786132 ]
 [ 0.06679175]
 [-0.01417918]
 [-0.04847382]
 [-0.02273274]
 [ 0.0234747 ]
 [ 0.0135873 ]
 [ 0.02612169]
 [ 0.0870638 ]
 [-0.06221372]
 [ 0.03502057]
 [ 0.26890012]
 [-0.01278494]
 [ 0.12956673]
 [ 0.14711362]
 [ 0.09046103]
 [-0.08757859]
 [-0.03575409]
 [ 0.04343987]
 [ 0.02805838]
 [-0.01557601]
 [-0.03532841]
 [ 0.00037178]
 [ 0.12066274]
 [-0.07047026]
 [ 0.06735499]
 [-0.03272228]
 [ 0.04303417]
 [-0.05351811]
 [ 0.02660246]
 [ 0.0399237 ]
 [-0.04925197]
 [-0.10282991]
 [ 0.02667252]
 [ 0.06143729]
 [-0.23752509]
 [-0.03276432]
 [-0.11645921]
 [ 0.27059597]
 [-0.06061505]
 [-0.10862189]
 [-0.09345131]
 [-0.0575947 ]
 [-0.12989435]
 [ 0.22773806]
 [-0.08629879]
 [ 0.09614